# 01 - Extraction and Cleaning

**Project:** AI-Powered Chatbot for Data Engineering Students (FYP, UiTM)
**Phase 6, Step 1:** Knowledge Base Preparation - extract text from the raw sources and clean it.

## What this notebook does
1. Reads the rules for each of the 7 sources (S01-S07) from `config/settings.py`. Nothing about paths, page ranges or cleaning rules is hardcoded here.
2. Extracts the text from the raw files in `data/raw/` (**read-only**: the raw files are never modified).
3. Applies the cleaning rules (strip front matter, header lines, bullet characters, wiki links, ...).
4. Prints a **BEFORE** (raw) and **AFTER** (cleaned) sample for every source.
5. Writes the cleaned text to `data/cleaned/` and a manifest file, `extraction_manifest.json`.

## What this notebook does NOT do
- No chunking (that is `02_chunking.ipynb`).
- **S03 extractor is NOT decided yet.** Section 5 compares PyMuPDF and pdfplumber side by side; both candidate outputs are written and the decision is made after review.

## Output format
- **PDF sources** become one `.txt` file per source. Each page starts with a marker line `=== page N ===` so that the page number can be kept as metadata during chunking.
- **Markdown / code sources (S05, S06)** keep their original folder structure under `data/cleaned/S05/` and `data/cleaned/S06/`, one file per original file.

## 1. Setup

In [1]:
import sys
import re
import json
import math
import shutil
import zipfile
import logging
import textwrap
from itertools import zip_longest
from pathlib import Path

import pandas as pd
import pdfplumber
import pymupdf

# The notebook lives in notebooks/, so walk up until the folder that holds config/ is found.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import settings  # all paths and rules come from here

# pdfplumber prints harmless "FontBBox" warnings for some PDFs; hide them.
logging.getLogger("pdfminer").setLevel(logging.ERROR)

# Display-only settings for this notebook (not part of the pipeline rules).
SAMPLE_CHARS = 400     # how many characters to show in BEFORE / AFTER samples
LONG_TOKEN_LEN = 20    # a "word" longer than this is suspicious (merged words) in the S03 comparison

settings.CLEANED_DIR.mkdir(parents=True, exist_ok=True)
summary_rows = []      # one row per output, printed as a table at the end
manifest = []          # one entry per output, saved to extraction_manifest.json

print("Project root :", settings.PROJECT_ROOT)
print("Raw data     :", settings.RAW_DIR)
print("Cleaned data :", settings.CLEANED_DIR)
print("Sources      :", ", ".join(settings.SOURCES))

Project root : C:\Users\thaf\OneDrive\Degree\FYP\fyp-data-engineering-chatbot
Raw data     : C:\Users\thaf\OneDrive\Degree\FYP\fyp-data-engineering-chatbot\data\raw
Cleaned data : C:\Users\thaf\OneDrive\Degree\FYP\fyp-data-engineering-chatbot\data\cleaned
Sources      : S01, S02, S03, S04, S05, S06, S07


## 2. Helper functions
Small, single-purpose functions used by every section below.

In [2]:
def page_numbers(total_pages, page_ranges):
    """Return the 1-based page numbers to process.

    page_ranges is a list of (first, last) tuples from settings.SOURCES, or None for all pages.
    """
    if page_ranges is None:
        return list(range(1, total_pages + 1))
    wanted = []
    for first, last in page_ranges:
        wanted.extend(range(first, min(last, total_pages) + 1))
    return wanted


def extract_pdf_pages(pdf_path, extractor, page_ranges):
    """Extract raw text page by page. Returns a list of (page_number, raw_text).

    extractor is "pdfplumber" or "pymupdf". The PDF is opened read-only.
    """
    pages = []
    if extractor == "pdfplumber":
        with pdfplumber.open(pdf_path) as pdf:
            for number in page_numbers(len(pdf.pages), page_ranges):
                pages.append((number, pdf.pages[number - 1].extract_text() or ""))
    elif extractor == "pymupdf":
        with pymupdf.open(pdf_path) as doc:
            for number in page_numbers(len(doc), page_ranges):
                pages.append((number, doc[number - 1].get_text("text") or ""))
    else:
        raise ValueError(f"Unknown extractor: {extractor}")
    return pages


def strip_text_patterns(text, source):
    """Remove multi-line blocks matching the regexes in settings ('strip_text_patterns')."""
    for pattern in source.get("strip_text_patterns", []):
        text = re.sub(pattern, "", text)
    return text


def apply_source_rules(text, source):
    """Apply the source-specific rules from settings: strip_text_patterns, strip_line_patterns, strip_characters."""
    text = strip_text_patterns(text, source)
    for pattern in source.get("strip_line_patterns", []):
        regex = re.compile(pattern)
        text = "\n".join(line for line in text.splitlines() if not regex.match(line))
    for character in source.get("strip_characters", []):
        text = text.replace(character, "")
    if source.get("collapse_repeated_lines"):              # a line identical to the previous line is dropped
        kept_lines = []
        for line in text.splitlines():
            if kept_lines and line.strip() and line.strip() == kept_lines[-1].strip():
                continue
            kept_lines.append(line)
        text = "\n".join(kept_lines)
    return text


def normalize_pdf_text(text):
    """Tidy whitespace in PDF text: collapse repeated spaces, trim lines, limit blank lines."""
    text = re.sub(r"[ \t]+", " ", text)
    text = "\n".join(line.strip() for line in text.splitlines())
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_pdf_pages(raw_pages, source):
    """Clean every extracted page. Pages shorter than MIN_PAGE_CHARACTERS are dropped as empty.

    Returns (all_records, kept_records); each record keeps raw / rule-cleaned / final text.
    """
    records = []
    for number, raw in raw_pages:
        ruled = apply_source_rules(raw, source)
        records.append({"page": number, "raw": raw, "ruled": ruled, "clean": normalize_pdf_text(ruled)})
    kept = [r for r in records if len(r["clean"]) >= settings.MIN_PAGE_CHARACTERS]
    return records, kept


def write_pdf_text(source_id, kept_records, suffix=""):
    """Write cleaned PDF pages to data/cleaned/<id><suffix>.txt with '=== page N ===' markers."""
    out_path = settings.CLEANED_DIR / f"{source_id}{suffix}.txt"
    body = "\n\n".join(f"=== page {r['page']} ===\n{r['clean']}" for r in kept_records)
    out_path.write_text(body, encoding="utf-8")
    return out_path


def first_difference(a, b):
    """Index of the first character where a and b differ (None if identical)."""
    if a == b:
        return None
    return next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))


def pick_sample(records):
    """Pick the page that best shows the cleaning rules at work.

    Prefers a page where a rule changed text within the first SAMPLE_CHARS characters;
    otherwise the first page with enough text.
    """
    candidates = []
    for r in records:
        diff = first_difference(r["raw"], r["ruled"])
        if diff is not None and diff < SAMPLE_CHARS:
            candidates.append((diff, r))
    if candidates:
        return min(candidates, key=lambda c: c[0])[1], True
    # No rule changed anything near the top of a page: show a page from the middle of the
    # source instead (more representative than a chapter title page).
    full_pages = [r for r in records if len(r["clean"]) >= SAMPLE_CHARS]
    if full_pages:
        return full_pages[len(full_pages) // 2], False
    return records[0], False


def repeated_lines(kept_records, min_share=0.15, top=6):
    """Diagnostic: lines that appear on at least min_share of the pages (running headers/footers).

    Digits are replaced by '#' before counting, so a header whose page number changes
    ("30 1 • Title", "32 1 • Title") is still recognised as the same repeated line.
    """
    counts = {}
    for r in kept_records:
        for line in {re.sub(r"\d+", "#", l.strip()) for l in r["clean"].splitlines() if l.strip()}:
            counts[line] = counts.get(line, 0) + 1
    threshold = max(3, min_share * len(kept_records))
    return sorted(((n, l) for l, n in counts.items() if n >= threshold), reverse=True)[:top]


def count_remaining_matches(text, source):
    """Count how often text still matches a strip rule from settings (every value should be 0)."""
    left = {}
    for pattern in source.get("strip_line_patterns", []):
        left[pattern] = sum(1 for line in text.splitlines() if re.match(pattern, line))
    for pattern in source.get("strip_text_patterns", []):
        left[pattern] = len(re.findall(pattern, text))
    for character in source.get("strip_characters", []):
        left[repr(character)] = text.count(character)
    if source.get("collapse_repeated_lines"):
        lines = text.splitlines()
        left["consecutive duplicate lines"] = sum(1 for a, b in zip(lines, lines[1:]) if a.strip() and a.strip() == b.strip())
    return left


def show_before_after(label, before, after, note="", from_end=False):
    """Print SAMPLE_CHARS characters of the raw text and of the cleaned text (start, or end if from_end)."""
    cut = (lambda t: "..." + t[-SAMPLE_CHARS:]) if from_end else (lambda t: t[:SAMPLE_CHARS])
    print("=" * 90)
    print(label, f"({note})" if note else "")
    print("-" * 90)
    print("BEFORE (raw extracted text):")
    print(cut(before))
    print("-" * 90)
    print("AFTER (cleaned text):")
    print(cut(after))
    print("=" * 90)


def add_summary(source_id, fmt, processed, characters, outputs, note=""):
    """Record one row for the final summary table and one entry for the manifest."""
    summary_rows.append({
        "source": source_id,
        "format": fmt,
        "pages/files processed": processed,
        "characters extracted": characters,
        "output file(s) written": outputs,
    })

## 3. PDF textbooks: S01, S02, S04 and S07
These four sources use the same procedure: extract with the extractor named in `settings.py`, keep only the configured page ranges, clean, and write one `.txt` file.

Each of these books prints a **running page header** on every page (for example `30 1 • What Are Data and Data Science?` in S02, `Chapter 5 Data Modelling | 20` in S01) with a page number that changes from page to page, and the OpenStax books also print `Access for free at openstax.org` on about half of the pages. `settings.py` has line-pattern rules for them. If left in, these lines would end up in the middle of chunks. S07 is handled in its own section below. For every source the cell prints a **rule check**: how many text fragments still match a cleaning rule after cleaning (all should be 0).

In [3]:
def process_pdf_source(source_id):
    """Run the full extract -> clean -> write procedure for one PDF source and show a sample."""
    source = settings.SOURCES[source_id]
    pdf_path = settings.RAW_DIR / source["path"]

    raw_pages = extract_pdf_pages(pdf_path, source["extractor"], source.get("page_ranges"))
    records, kept = clean_pdf_pages(raw_pages, source)
    out_path = write_pdf_text(source_id, kept)

    sample, rule_applied = pick_sample(records)
    note = f"page {sample['page']}, " + ("a cleaning rule changed this page" if rule_applied else "mid-source page, whitespace tidying only")
    show_before_after(f"{source_id} - {source['name']}", sample["raw"], sample["clean"], note)

    chars = sum(len(r["clean"]) for r in kept)
    print(f"{source_id}: extractor={source['extractor']}, pages in range={len(records)}, "
          f"empty pages dropped={len(records) - len(kept)}, characters={chars:,}")
    print(f"Written: {out_path.relative_to(settings.PROJECT_ROOT)}")
    left = count_remaining_matches("\n".join(r["clean"] for r in kept), source)
    print("Rule check, matches remaining after cleaning:",
          f"ALL ZERO ({len(left)} rules checked)" if not any(left.values()) else {k[:40]: v for k, v in left.items() if v})
    repeats = repeated_lines(kept)
    print("Lines repeated on >=15% of pages (possible running headers/footers):",
          [f"{n}x {line[:50]!r}" for n, line in repeats] if repeats else "none")
    print()

    add_summary(source_id, "PDF", f"{len(kept)} pages ({len(records)} in range)", chars,
                str(out_path.relative_to(settings.PROJECT_ROOT)))
    manifest.append({"source_id": source_id, "source_type": source["source_type"], "format": "pdf",
                     "extractor": source["extractor"], "output": out_path.name,
                     "pages": len(kept), "characters": chars, "page_markers": True})
    return records, kept

In [4]:
s01_records, s01_kept = process_pdf_source("S01")

S01 - BCcampus - Database Design (2nd edition, Watt) (page 22, a cleaning rule changed this page)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
2. Key Terms
3. Exercises
13 | Chapter 3 Characteristics and Benefits of a Database
------------------------------------------------------------------------------------------
AFTER (cleaned text):
2. Key Terms
3. Exercises
S01: extractor=pdfplumber, pages in range=143, empty pages dropped=4, characters=192,092
Written: data\cleaned\S01.txt
Rule check, matches remaining after cleaning: ALL ZERO (4 rules checked)
Lines repeated on >=15% of pages (possible running headers/footers): none



In [5]:
s02_records, s02_kept = process_pdf_source("S02")

S02 - OpenStax - Principles of Data Science (page 20, a cleaning rule changed this page)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
10 1 • What Are Data and Data Science?
In its earliest days, the work of data science was spread across multiple disciplines, including statistics,
mathematics, computer science, and social science. It was commonly believed that the job of data collection,
management, and analysis would be carried out by different types of experts, with each job independent of
one another. To be more specific, dat
------------------------------------------------------------------------------------------
AFTER (cleaned text):
In its earliest days, the work of data science was spread across multiple disciplines, including statistics,
mathematics, computer science, and social science. It was commonly believed that the job of data collection,
management, and analysis would be carried out by different 

In [6]:
s04_records, s04_kept = process_pdf_source("S04")

S04 - OpenStax - Introduction to Computer Science (page 370, a cleaning rule changed this page)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
360 8 • Data Management
As an example, TechWorks is a start-up company that is 100% committed to leveraging innovative technologies
as part of its repeatable business model and as a business growth facilitator. TechWorks has multiple
departments including human resources, finance, sales, marketing, operation management, and information
technology. Each department has its own information system and
------------------------------------------------------------------------------------------
AFTER (cleaned text):
As an example, TechWorks is a start-up company that is 100% committed to leveraging innovative technologies
as part of its repeatable business model and as a business growth facilitator. TechWorks has multiple
departments including human resources, finance, sales, marke

In [7]:
# Explicit check for the OpenStax footer in the RAW pages and in the cleaned pages
for source_id, records in [("S02", s02_records), ("S04", s04_records)]:
    footer = "Access for free at openstax.org"
    raw_pages_with = sum(footer in r["raw"] for r in records)
    clean_pages_with = sum(footer in r["clean"] for r in records)
    print(f"{source_id}: pages with the footer  raw = {raw_pages_with} of {len(records)}   cleaned = {clean_pages_with}")

S02: pages with the footer  raw = 47 of 94   cleaned = 0
S04: pages with the footer  raw = 35 of 70   cleaned = 0


### S07 - removing web-page junk
S07 is a browser printout. `settings.py` removes four kinds of junk:
- the print header `10/6/26, 2:20 PM 1.4 Data Quality ...` (line pattern),
- the Pressbooks navigation block `Home Read Sign / Search in book ... / Want to create or adapt books like this? ...` (text pattern),
- the footer `https://pressbooks.pub/... 1/9` (line pattern),
- the platform lines at the end of the last page, `Powered by Pressbooks` and `Pressbooks User Guide | Pressbooks Directory | Contact` (line patterns). The book's own copyright / CC BY-NC-SA sentence above them is kept on purpose.

The BEFORE sample should show them and the AFTER sample should not. The checks below count how many matches remain in the cleaned text (should be 0).

In [8]:
s07_records, s07_kept = process_pdf_source("S07")

s07_rules = settings.SOURCES["S07"]
clean_text = "\n".join(r["clean"] for r in s07_kept)
print("Junk still present in the cleaned S07 text (should all be 0):")
for pattern in s07_rules["strip_line_patterns"]:
    left = sum(1 for line in clean_text.splitlines() if re.match(pattern, line))
    print(f"  line pattern {pattern[:55]!r:60} -> {left}")
for pattern in s07_rules["strip_text_patterns"]:
    print(f"  text pattern {pattern[:55]!r:60} -> {len(re.findall(pattern, clean_text))}")

# Show the END of the last page: the footer URL and page counter must be gone
last = s07_kept[-1]
show_before_after("S07 - end of the last page", last["raw"], last["clean"], f"page {last['page']}: footer removed", from_end=True)

S07 - Luna Reyes et al. - Data Analytics for Public Policy, section 1.4 (page 1, a cleaning rule changed this page)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
10/6/26, 2:20 PM 1.4 Data Quality and Data Types – Data Analytics for Public Policy and Management
Home Read Sign
Search in book …
in
Want to create or adapt books like this? Learn more⧉ about how Pressbooks
supports open publishing practices.
DATA ANALYTICS FOR PUBLIC POLICY AND MANAGEMENT
CONTENTS
1.4 Data Quality and Data Types
Understanding data and data quality are basic requirements to work 
------------------------------------------------------------------------------------------
AFTER (cleaned text):
DATA ANALYTICS FOR PUBLIC POLICY AND MANAGEMENT
CONTENTS
1.4 Data Quality and Data Types
Understanding data and data quality are basic requirements to work as a data sci-
entist. There are many different ways of thinking about data and classifying da

## 4. S05 - Data Engineering Wiki (Markdown in a ZIP)
Rules from `settings.py`:
- read only the `.md` files, skip `README.md` and `LICENSE`;
- remove the front matter block (`--- ... ---`) at the top of each page;
- remove the wiki footer block (GitHub edit/copy links and feedback links) from its start to the end of the file. Most files start it with a `%% wiki footer: ... %%` line; a few only have the `## This note in GitHub` heading, so there are two patterns;
- resolve `[[wiki links]]` to plain text (`[[Target|Alias]]` becomes `Alias`) and remove `![[image embeds]]`.

The ZIP is read in memory; nothing is extracted into `data/raw/`.

In [9]:
FRONTMATTER_RE = re.compile(r"\A---\s*\n.*?\n---\s*(?:\n|\Z)", re.DOTALL)
EMBED_RE = re.compile(r"!\[\[[^\]]*\]\]")        # ![[image.png|100]]
WIKILINK_RE = re.compile(r"\[\[([^\]]*)\]\]")     # [[Target]] or [[Target|Alias]]


def list_zip_files(zip_path):
    """Return [(ZipInfo, relative_path)] for every file, dropping the ZIP's single top-level folder."""
    archive = zipfile.ZipFile(zip_path)
    entries = []
    for info in archive.infolist():
        if info.is_dir():
            continue
        parts = info.filename.split("/")
        if len(parts) > 1:
            entries.append((info, "/".join(parts[1:])))
    return archive, entries


def wiki_link_to_text(match):
    """Turn the inside of a [[wiki link]] into readable text (alias if present, else the target name)."""
    target, _, alias = match.group(1).partition("|")
    alias = alias.strip()
    if alias:
        return alias
    target = target.rstrip("\\").split("#")[0]     # drop '#Heading' part and any escape character
    return target.split("/")[-1].strip()            # drop folder prefix such as 'Concepts/'


def clean_markdown(text, source):
    """Apply the Markdown cleaning rules chosen in settings (front matter, wiki links) and tidy whitespace."""
    text = text.replace("\r\n", "\n")
    if source.get("strip_frontmatter"):
        text = FRONTMATTER_RE.sub("", text, count=1)
    text = strip_text_patterns(text, source)        # e.g. the S05 wiki footer block
    if source.get("resolve_wiki_links"):
        text = EMBED_RE.sub("", text)
        text = WIKILINK_RE.sub(wiki_link_to_text, text)
    text = "\n".join(line.rstrip() for line in text.splitlines())
    return re.sub(r"\n{3,}", "\n\n", text).strip() + "\n"


def reset_output_folder(folder):
    """Delete a previously generated output folder (only ever inside data/cleaned/) and recreate it."""
    assert settings.CLEANED_DIR.resolve() in folder.resolve().parents, "refusing to touch a folder outside data/cleaned"
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True)

In [10]:
s05 = settings.SOURCES["S05"]
archive, entries = list_zip_files(settings.RAW_DIR / s05["path"])

selected = [(info, rel) for info, rel in entries
            if Path(rel).suffix.lower() in s05["include_extensions"]
            and Path(rel).name not in s05["exclude_filenames"]]
print(f"S05: {len(entries)} files in the ZIP, {len(selected)} selected")

out_dir = settings.CLEANED_DIR / "S05"
reset_output_folder(out_dir)

s05_chars = 0
sample_shown = footer_shown = False
footer_files_before = 0
for info, rel in selected:
    raw = archive.read(info).decode("utf-8", errors="replace")
    cleaned = clean_markdown(raw, s05)
    target = out_dir / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(cleaned, encoding="utf-8")
    s05_chars += len(cleaned)
    footer_files_before += "%% wiki footer" in raw
    manifest.append({"source_id": "S05", "source_type": s05["source_type"], "format": "markdown",
                     "original_path": rel, "output": f"S05/{rel}", "characters": len(cleaned)})
    # Show the START of the first file that has front matter AND wiki links
    if not sample_shown and raw.startswith("---") and "[[" in raw:
        show_before_after(f"S05 - {rel}", raw, cleaned, "start of file: front matter removed, wiki links resolved")
        sample_shown = True
    # Show the END of the first file that has the wiki footer
    if not footer_shown and "%% wiki footer" in raw:
        show_before_after(f"S05 - {rel}", raw, cleaned, "end of file: wiki footer removed", from_end=True)
        footer_shown = True

print(f"Files that had a wiki footer in the raw text: {footer_files_before} of {len(selected)}")

print(f"S05: wrote {len(selected)} files, {s05_chars:,} characters")
add_summary("S05", "Markdown (ZIP)", f"{len(selected)} files", s05_chars, f"{len(selected)} files in data/cleaned/S05/")

cleaned_texts = {p: p.read_text(encoding="utf-8") for p in out_dir.rglob("*.md")}
print("Cleaned S05 files that still contain 'wiki footer'  :", sum("wiki footer" in t for t in cleaned_texts.values()))
print("Cleaned S05 files that still contain 'This note in GitHub':", sum("This note in GitHub" in t for t in cleaned_texts.values()))
print("Cleaned S05 files that still contain 'git-footer'   :", sum("git-footer" in t for t in cleaned_texts.values()))
remaining = {}
for text in cleaned_texts.values():
    for pattern, n in count_remaining_matches(text, s05).items():
        remaining[pattern] = remaining.get(pattern, 0) + n
print("Rule check, matches remaining across all S05 files:", "ALL ZERO" if not any(remaining.values()) else remaining)
message_broker = cleaned_texts[out_dir / "Concepts" / "Data Processing" / "Message Broker.md"]
print("Message Broker.md now ends with:", repr(message_broker[-120:]))
print("Cleaned S05 files that still start with front matter:", sum(t.startswith("---") for t in cleaned_texts.values()))
print("Cleaned S05 files that still contain '[[' :",
      [(p.relative_to(out_dir).as_posix()) for p, t in cleaned_texts.items() if "[[" in t])

S05: 79 files in the ZIP, 76 selected
S05 - Guides/Testing Your Data Pipeline.md (start of file: front matter removed, wiki links resolved)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
---
Aliases: []
Tags: [evergreen]
publish: true
---

## Why Is Testing Important?

Writing tests have a few key benefits for data engineers. They can help you sleep better at night knowing you're less likely to get called into an emergency for a data issue. They can be used to help yourself and stakeholders get on the same page and understand the data you're working with. And one of the most impor
------------------------------------------------------------------------------------------
AFTER (cleaned text):
## Why Is Testing Important?

Writing tests have a few key benefits for data engineers. They can help you sleep better at night knowing you're less likely to get called into an emergency for a data issue. They can be used to h

## 5. S03 - Helsinki lecture slides
S03 was the problem source: with pdfplumber the words of many slides run together. **Decision: PyMuPDF** (set in `settings.py`). This section keeps the evidence for that decision, then produces the final output.

### 5a. Evidence: the two extractors on the same pages
Both extractors listed in `settings.SOURCES["S03"]["extractors_compared"]` are run on the same pages and compared side by side and with a simple measure: how many "words" are suspiciously long (merged words) per 1000 words.

In [11]:
s03 = settings.SOURCES["S03"]
s03_candidates = s03["extractors_compared"]                  # taken from settings.py
print("Extractors compared:", s03_candidates, "| chosen:", s03["extractor"])

s03_results = {}
for name in s03_candidates:
    raw_pages = extract_pdf_pages(settings.RAW_DIR / s03["path"], name, s03.get("page_ranges"))
    records, kept = clean_pdf_pages(raw_pages, s03)       # same cleaning rules for both
    s03_results[name] = {"records": records, "kept": kept}


def suspicious_tokens(text):
    """Words longer than LONG_TOKEN_LEN characters that are not URLs, paths or numbers (likely merged words)."""
    tokens = re.findall(r"\S+", text)
    bad = [t for t in tokens if len(t) > LONG_TOKEN_LEN and not re.search(r"[\d/@:_\\-]|https?|www", t)]
    return tokens, bad


rows = []
for name, res in s03_results.items():
    all_text = "\n".join(r["clean"] for r in res["kept"])
    tokens, bad = suspicious_tokens(all_text)
    rows.append({"extractor": name, "words": len(tokens), "suspicious long words": len(bad),
                 "per 1000 words": round(1000 * len(bad) / max(len(tokens), 1), 1),
                 "avg word length": round(sum(map(len, tokens)) / max(len(tokens), 1), 2)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nLower 'suspicious long words' = fewer merged words.")
for name, res in s03_results.items():
    _, bad = suspicious_tokens("\n".join(r["clean"] for r in res["kept"]))
    print(f"Examples from {name}:", bad[:5])

Extractors compared: ['pymupdf', 'pdfplumber'] | chosen: pymupdf


 extractor  words  suspicious long words  per 1000 words  avg word length
   pymupdf   8037                     16             2.0             5.63
pdfplumber   7084                    125            17.6             6.86

Lower 'suspicious long words' = fewer merged words.
Examples from pymupdf: ['mapPartitionsWithIndex(func)', 'sample(withReplacement,', 'intersection(otherDataset)', 'distinct([numTasks]))', 'groupByKey([numTasks])']
Examples from pdfplumber: ['Cloudcomputingisamodelforenablingconvenient,', 'computingresources(e.g.,networks,servers,storage,', 'applications,andservices)thatcanberapidlyprovisioned', 'andreleasedwithminimalmanagementeffortorservice', 'Thiscloudmodelpromotesavailability']


In [12]:
def side_by_side(title_left, text_left, title_right, text_right, width=58, max_lines=26):
    """Print two texts in two columns so differences in spacing are easy to see."""
    def wrap(text):
        lines = []
        for line in text.splitlines():
            lines.extend(textwrap.wrap(line, width) or [""])
        return lines[:max_lines]

    print(f"{title_left:<{width}} | {title_right}")
    print("-" * width + "-+-" + "-" * width)
    for left, right in zip_longest(wrap(text_left), wrap(text_right), fillvalue=""):
        print(f"{left:<{width}} | {right}")


first, second = s03_candidates[0], s03_candidates[1]
pages_a = {r["page"]: r for r in s03_results[first]["records"]}
pages_b = {r["page"]: r for r in s03_results[second]["records"]}
common = sorted(set(pages_a) & set(pages_b))
worst_page = max(common, key=lambda p: len(suspicious_tokens(pages_b[p]["clean"])[1]))   # worst page for the second extractor

print("=" * 125)
print(f"S03 PAGE {worst_page} (the page with the most merged words under {second})")
print("=" * 125)
side_by_side(first.upper(), pages_a[worst_page]["clean"], second.upper(), pages_b[worst_page]["clean"])

S03 PAGE 39 (the page with the most merged words under pdfplumber)
PYMUPDF                                                    | PDFPLUMBER
-----------------------------------------------------------+-----------------------------------------------------------
MapReduce and Functional Programming                       | MapReduce and Functional Programming
Based on the functional programming in the large:          | 
User is only allowed to write side-effect free functions   | Based on the functional programming in the large:
“Map” and “Reduce”                                         | 
Re-execution is used for fault tolerance. If a computer    | Userisonlyallowedtowriteside-effectfreefunctions
executing a Map or a Reduce task fails to produce a result | “Map”and“Reduce”
due to hardware failure, the task will be re-executed on   | 
another computer                                           | Re-executionisusedforfaulttolerance. Ifacomputer
Side effects in functions would make this imposs

### 5b. Final S03 output with the chosen extractor
`process_pdf_source` uses the extractor from `settings.py` and applies the S03 rules: strip the `▶` bullets, the repeated header `Big Data Platforms (5 ECTS)`, the slide counter such as `12/44`, and collapse a slide title that PyMuPDF prints twice in a row into one line. The BEFORE sample should show the header and counter; the AFTER sample should not.

Earlier candidate files (`S03_pymupdf.txt`, `S03_pdfplumber.txt`) are deleted so only the final `S03.txt` remains.

In [13]:
for stale in settings.CLEANED_DIR.glob("S03_*.txt"):     # generated candidate outputs from the comparison stage
    stale.unlink()
    print("Deleted old candidate file:", stale.name)

s03_records, s03_kept = process_pdf_source("S03")

s03_text = "\n".join(r["clean"] for r in s03_kept)
print("Junk still present in the cleaned S03 text (should all be 0):")
for pattern in s03["strip_line_patterns"]:
    print(f"  line pattern {pattern!r:45} -> {sum(1 for line in s03_text.splitlines() if re.match(pattern, line))}")
print("  '▶' characters                                       ->", s03_text.count("\u25b6"))

S03 - University of Helsinki - Big Data Platforms (Heljanko, 2026) (page 1, a cleaning rule changed this page)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
Big Data Platforms (5 ECTS)
12/44
Cloud Computing
▶The NIST Definition of Cloud Computing
Authors: Peter Mell and Tim Grance
SP 800-145
https://csrc.nist.gov/publications/detail/sp/
800-145/final
▶Cloud computing is a model for enabling convenient,
on-demand network access to a shared pool of configurable
computing resources (e.g., networks, servers, storage,
applications, and services) that can b
------------------------------------------------------------------------------------------
AFTER (cleaned text):
Cloud Computing
The NIST Definition of Cloud Computing
Authors: Peter Mell and Tim Grance
SP 800-145
https://csrc.nist.gov/publications/detail/sp/
800-145/final
Cloud computing is a model for enabling convenient,
on-demand network access to a shared pool 

## 6. S06 - Data Engineering Zoomcamp (Markdown, SQL and Python in a ZIP)
Rules from `settings.py`:
- keep only the seven module folders `01-...` to `07-...`; skip `cohorts/`, `.tmp/`, `.github/`, `projects/`, `scripts/` and images;
- skip `07-streaming/extras/` (duplicate/example material with few comments);
- skip `07-streaming/code/live/`: a workshop snapshot whose files are exact copies or older variants of `07-streaming/code/src/`, which is kept as the single copy;
- keep `.md`, `.sql` and `.py` files; `.md` is labelled `course_notes`, `.sql` and `.py` are labelled `code`;
- strip the front matter (`prev_url`, `next_url`, ...) from the Markdown files.

After extraction, the remaining code files of `07-streaming` (now only `07-streaming/code/`) are measured again.

In [14]:
s06 = settings.SOURCES["S06"]
archive6, entries6 = list_zip_files(settings.RAW_DIR / s06["path"])

include_ext = set(s06["include_extensions"])
exclude_ext = set(s06["exclude_extensions"])
skipped = {"not in a module folder": 0, "excluded folder": 0, "excluded path (extras/, code/live/)": 0, "extension not wanted": 0}
selected6 = []
for info, rel in entries6:
    parts = rel.split("/")
    suffix = Path(rel).suffix.lower()
    if rel.startswith(tuple(s06["exclude_paths"])):
        skipped["excluded path (extras/, code/live/)"] += 1
    elif any(p in s06["exclude_folders"] for p in parts[:-1]):
        skipped["excluded folder"] += 1
    elif parts[0] not in s06["include_folders"]:
        skipped["not in a module folder"] += 1
    elif suffix not in include_ext or suffix in exclude_ext:
        skipped["extension not wanted"] += 1
    else:
        selected6.append((info, rel))

print(f"S06: {len(entries6)} files in the ZIP -> {len(selected6)} selected")
print("Skipped:", skipped)

S06: 954 files in the ZIP -> 137 selected
Skipped: {'not in a module folder': 20, 'excluded folder': 197, 'excluded path (extras/, code/live/)': 82, 'extension not wanted': 518}


In [15]:
out_dir6 = settings.CLEANED_DIR / "S06"
reset_output_folder(out_dir6)

s06_records = []
sample_shown = False
for info, rel in selected6:
    raw = archive6.read(info).decode("utf-8", errors="replace")
    suffix = Path(rel).suffix.lower()
    source_type = s06["source_type_by_extension"][suffix]
    if suffix == ".md":
        cleaned = clean_markdown(raw, s06)           # strip front matter, tidy whitespace
    else:
        cleaned = raw.replace("\r\n", "\n")          # code: keep exactly as written (indentation matters)
    target = out_dir6 / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(cleaned, encoding="utf-8")

    s06_records.append({"module": rel.split("/")[0], "file": rel, "extension": suffix,
                        "source_type": source_type, "characters": len(cleaned)})
    manifest.append({"source_id": "S06", "source_type": source_type, "format": suffix.lstrip("."),
                     "original_path": rel, "output": f"S06/{rel}", "characters": len(cleaned)})
    if not sample_shown and suffix == ".md" and raw.startswith("---"):
        show_before_after(f"S06 - {rel}", raw, cleaned, "front matter removed")
        sample_shown = True

s06_df = pd.DataFrame(s06_records)
s06_total = int(s06_df["characters"].sum())
print(f"S06: wrote {len(s06_df)} files, {s06_total:,} characters")
add_summary("S06", "Markdown + code (ZIP)", f"{len(s06_df)} files", s06_total, f"{len(s06_df)} files in data/cleaned/S06/")

S06 - 01-docker-terraform/01-introduction.md (front matter removed)
------------------------------------------------------------------------------------------
BEFORE (raw extracted text):
---
next_url: 02-virtual-environment.md
---
# Introduction to Docker

Docker is a _containerization software_ that allows us to isolate software in a similar way to virtual machines but in a much leaner way.

A Docker image is a _snapshot_ of a container that we can define to run our software, or in this case our data pipelines. By exporting our Docker images to Cloud providers such as Amazon Web 
------------------------------------------------------------------------------------------
AFTER (cleaned text):
# Introduction to Docker

Docker is a _containerization software_ that allows us to isolate software in a similar way to virtual machines but in a much leaner way.

A Docker image is a _snapshot_ of a container that we can define to run our software, or in this case our data pipelines. By exportin

S06: wrote 137 files, 512,229 characters

### S06 - duplicate check
No file under an excluded path may be written, and no two output files may have identical content.

In [16]:
import hashlib

written = [p for p in out_dir6.rglob("*") if p.is_file()]
under_live = [p for p in written if "07-streaming/code/live/" in p.as_posix()]
under_extras = [p for p in written if "07-streaming/extras/" in p.as_posix()]
by_hash = {}
for p in written:
    by_hash.setdefault(hashlib.md5(p.read_bytes()).hexdigest(), []).append(p.relative_to(out_dir6).as_posix())
duplicate_groups = [files for files in by_hash.values() if len(files) > 1]

print("Files written for S06                       :", len(written))
print("Files written under 07-streaming/code/live/ :", len(under_live), "(should be 0)")
print("Files written under 07-streaming/extras/    :", len(under_extras), "(should be 0)")
print("Groups of files with identical content      :", len(duplicate_groups), "(should be 0)")
for group in duplicate_groups:
    print("   ", group)
print("producer_realtime.py copies kept            :", [p.relative_to(out_dir6).as_posix() for p in written if p.name == "producer_realtime.py"])
print("models.py copies kept                       :", [p.relative_to(out_dir6).as_posix() for p in written if p.name == "models.py"])

Files written for S06                       : 137
Files written under 07-streaming/code/live/ : 0 (should be 0)
Files written under 07-streaming/extras/    : 0 (should be 0)
Groups of files with identical content      : 0 (should be 0)
producer_realtime.py copies kept            : ['07-streaming/code/src/producers/producer_realtime.py']
models.py copies kept                       : ['07-streaming/code/src/models.py']


### S06 - how much do the code files contribute?
Characters and files per module and type. "Estimated chunks" is only a rough guide (characters divided by the chunk size and overlap from `settings.py`); the real chunking happens in notebook 02.

In [17]:
def estimated_chunks(characters):
    """Rough chunk count for one file using settings.CHUNK_SIZE and settings.CHUNK_OVERLAP."""
    step = settings.CHUNK_SIZE - settings.CHUNK_OVERLAP
    return max(1, math.ceil(max(characters - settings.CHUNK_OVERLAP, 1) / step))


s06_df["est_chunks"] = s06_df["characters"].apply(estimated_chunks)

table = (s06_df.groupby(["module", "source_type", "extension"])
         .agg(files=("file", "count"), characters=("characters", "sum"), est_chunks=("est_chunks", "sum"))
         .reset_index())
print(table.to_string(index=False))

by_type = s06_df.groupby("source_type").agg(files=("file", "count"), characters=("characters", "sum"),
                                            est_chunks=("est_chunks", "sum"))
by_type["% of S06 characters"] = (100 * by_type["characters"] / s06_total).round(1)
print("\nS06 by source_type:")
print(by_type.to_string())

                   module  source_type extension  files  characters  est_chunks
      01-docker-terraform         code       .py      1        2373           4
      01-docker-terraform course_notes       .md     18       52150          78
02-workflow-orchestration course_notes       .md     19       34247          55
        03-data-warehouse         code       .py      3       14202          22
        03-data-warehouse         code      .sql      3        5427           9
        03-data-warehouse course_notes       .md      8       47452          71
 04-analytics-engineering         code      .sql     11       11535          21
 04-analytics-engineering course_notes       .md     16      107151         158
        05-data-platforms course_notes       .md     11       36023          56
                 06-batch         code       .py      2        5203           8
                 06-batch course_notes       .md     21      125513         187
             07-streaming         code  



S06 by source_type:
              files  characters  est_chunks  % of S06 characters
source_type                                                     
code             28       53882          88                 10.5
course_notes    109      458347         691                 89.5


In [18]:
streaming_py = s06_df[(s06_df["module"] == "07-streaming") & (s06_df["extension"] == ".py")]
streaming_all = s06_df[s06_df["module"] == "07-streaming"]
streaming_md = streaming_all[streaming_all["extension"] == ".md"]

print("07-streaming .py files")
print(f"  files                    : {len(streaming_py)}")
print(f"  characters               : {streaming_py['characters'].sum():,}")
print(f"  estimated chunks         : {streaming_py['est_chunks'].sum()}")
print(f"  share of the 07 module   : {100 * streaming_py['characters'].sum() / streaming_all['characters'].sum():.1f}% of its characters")
print(f"  share of all of S06      : {100 * streaming_py['characters'].sum() / s06_total:.1f}% of its characters")
print(f"  for comparison, 07 .md   : {len(streaming_md)} files, {streaming_md['characters'].sum():,} characters, "
      f"{streaming_md['est_chunks'].sum()} estimated chunks")

print("\nLargest 07-streaming .py files:")
print(streaming_py.sort_values("characters", ascending=False).head(8)[["file", "characters"]].to_string(index=False))

print("\nSample of the first 07-streaming .py file (to judge if it is useful explanation or boilerplate):")
sample_file = streaming_py.sort_values("characters", ascending=False).iloc[0]["file"]
print(f"--- {sample_file} ---")
print((out_dir6 / sample_file).read_text(encoding="utf-8")[:600])

07-streaming .py files
  files                    : 8
  characters               : 15,142
  estimated chunks         : 24
  share of the 07 module   : 21.3% of its characters
  share of all of S06      : 3.0% of its characters
  for comparison, 07 .md   : 16 files, 55,811 characters, 86 estimated chunks

Largest 07-streaming .py files:
                                                file  characters
   07-streaming/code/src/job/aggregation_job_demo.py        3106
        07-streaming/code/src/job/aggregation_job.py        2752
07-streaming/code/src/producers/producer_realtime.py        2752
       07-streaming/code/src/job/pass_through_job.py        2605
07-streaming/code/src/consumers/consumer_postgres.py        1267
         07-streaming/code/src/producers/producer.py        1055
         07-streaming/code/src/consumers/consumer.py         931
                     07-streaming/code/src/models.py         674

Sample of the first 07-streaming .py file (to judge if it is useful explanat

## 7. Manifest and summary

In [19]:
manifest_path = settings.CLEANED_DIR / "extraction_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Manifest written: {manifest_path.relative_to(settings.PROJECT_ROOT)} ({len(manifest)} entries)\n")

summary = pd.DataFrame(summary_rows)
summary["characters extracted"] = summary["characters extracted"].map("{:,}".format)
summary = summary.rename(columns={"source": "source ID"})
print(summary.to_string(index=False))

Manifest written: data\cleaned\extraction_manifest.json (218 entries)

source ID                format    pages/files processed characters extracted         output file(s) written
      S01                   PDF 139 pages (143 in range)              192,092           data\cleaned\S01.txt
      S02                   PDF   93 pages (94 in range)              226,793           data\cleaned\S02.txt
      S04                   PDF   70 pages (70 in range)              203,528           data\cleaned\S04.txt
      S07                   PDF     9 pages (9 in range)               12,538           data\cleaned\S07.txt
      S05        Markdown (ZIP)                 76 files              129,591  76 files in data/cleaned/S05/
      S03                   PDF 133 pages (138 in range)               53,174           data\cleaned\S03.txt
      S06 Markdown + code (ZIP)                137 files              512,229 137 files in data/cleaned/S06/


## 8. Checklist before moving on
- [x] S03: PyMuPDF chosen (Section 5); running header and slide counter stripped; candidate files deleted.
- [x] S05: wiki footer block stripped.
- [x] S06: `07-streaming/extras/` excluded, `07-streaming/code/` kept.
- [x] S07: print header, navigation block, footer URL and Pressbooks platform lines stripped.
- [x] S02, S04: OpenStax running footer stripped.
- [x] S05: second footer pattern added (`## This note in GitHub`).
- [x] S06: `07-streaming/code/live/` excluded (duplicates of `code/src/`).
- [x] S01, S02, S04: running page headers (with changing page numbers) stripped.
- [ ] Review the "Lines repeated on >=15% of pages" output for every PDF source (digits are ignored, so changing page numbers no longer hide a repeated header).
- [ ] Spot-check one cleaned file per source in `data/cleaned/`.
- [ ] Then continue with `02_chunking.ipynb`.